In [6]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.3.5"
!pip install pycaret==3.3.2

  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-3.3.5.tar.gz (1.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl (28.9 MB)
Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl (9.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for lightgbm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      /private/var/folders/br/qhn11dt91tvcg9cqgczhjdlr0000gn/T/pip-build-env-pptcf3m5/overlay/lib/python3.9/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider remo

In [7]:
import pandas as pd
from pycaret.regression import *
from sklearn.model_selection import train_test_split

SEED = 128
df = pd.read_csv("../Dataset2_Demand/6_Elec_Demand_Final.csv")
df_sample = df.sample(50000, random_state=SEED)

train_df, val_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=SEED
)

reg = setup(
    data=train_df,
    target="england_wales_demand",
    session_id=SEED,
    fold=2,
    verbose=True
)

best_model = compare_models(sort="R2", n_select=1, turbo=True)

tuned_model = tune_model(
    best_model, 
    optimize="R2", 
    fold=5, 
    n_iter=20
)

final_model = finalize_model(tuned_model)

predictions = predict_model(final_model, data=val_df)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_preds(pred):
    y_true = pred["england_wales_demand"]
    y_pred = pred["prediction_label"]
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "R2": r2_score(y_true, y_pred)
    }

eval_final = eval_preds(predictions)
print(eval_final)

save_model(final_model, "../Model_ElecDemand/england_wales_demand_model")

,Description,Value
0,Session id,128
1,Target,england_wales_demand
2,Target type,Regression
3,Original data shape,"(35000, 15)"
4,Transformed data shape,"(35000, 15)"
5,Transformed train set shape,"(24500, 15)"
6,Transformed test set shape,"(10500, 15)"
7,Numeric features,13
8,Categorical features,1
9,Preprocess,True


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,2688.4580,12471334.3825,3531.3638,0.7602,0.1099,0.0850,0.5250
et,Extra Trees Regressor,2701.9818,13520615.2839,3676.9508,0.7400,0.1145,0.0856,0.2000
rf,Random Forest Regressor,2760.0985,13842000.1098,3720.3286,0.7339,0.1166,0.0878,0.4600
gbr,Gradient Boosting Regressor,3046.5527,15258862.7973,3906.1700,0.7066,0.1223,0.0968,0.5650
ada,AdaBoost Regressor,3366.3036,17611157.3424,4196.4777,0.6614,0.1356,0.1110,0.2500
knn,K Neighbors Regressor,3251.6669,18543410.5814,4305.9745,0.6435,0.1329,0.1025,0.0550
dt,Decision Tree Regressor,3158.0046,18662011.7719,4319.3914,0.6412,0.1358,0.1003,0.0500
lar,Least Angle Regression,3782.9649,22744911.1819,4769.1255,0.5627,0.1503,0.1217,0.0200
llar,Lasso Least Angle Regression,3794.2039,22859380.9591,4781.1057,0.5605,0.1506,0.1220,0.4450
lasso,Lasso Regression,3794.2039,22859381.0425,4781.1057,0.5605,0.1506,0.1220,0.0250


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1953.7980,7277065.2115,2697.6036,0.8618,0.0829,0.0607
1,1887.7579,6715256.3632,2591.3812,0.8700,0.0793,0.0588
2,1980.9416,7444048.8449,2728.3784,0.8554,0.0842,0.0620
3,1920.2339,7111768.7143,2666.7900,0.8636,0.0821,0.0601
4,1973.6699,7445131.9694,2728.5769,0.8572,0.0838,0.0615
Mean,1943.2802,7198654.2207,2682.5460,0.8616,0.0825,0.0606
Std,34.8322,271418.7413,51.0009,0.0051,0.0017,0.0011


Fitting 5 folds for each of 20 candidates, totalling 100 fits


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,1505.9335,4258006.1971,2063.4937,0.9195,0.0635,0.0470


[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
{'MAE': 1505.9334573390643, 'MSE': 4258006.197055934, 'RMSE': 2063.493687185869, 'R2': 0.9194780082058461}
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['settlement_period',
                                              'embedded_wind_generation',
                                              'embedded_wind_capacity',
                                              'embedded_solar_generation',
                                              'embedded_solar_capacity',
                                              'non_bm_stor',
                                              'pump_storage_pumping',
                                              'ifa2_flow', 'britned_flow',
                                              'moyle_flow', 'east_west_flow',
                                              'nemo_flow', 'year'],
                                     transformer=Simp...
                  TransformerWrapper(include=['settlement_date'],
                                     transformer=TargetEncoder(cols=['settlement_date